# 📦 Data Shipping 101: A Beginner's Guide to Apache Thrift

Welcome! If you've never heard of "Data Encoding" or "RPC," don't worry. Think of this notebook as a guide to running an **International Courier Service**.

### The Big Idea: The Language Barrier
Imagine you have a **Chinese Chef (a Python program)** and a **German Customer (a Java program)**. They need to exchange recipes, but they don't speak the same language, and sending huge, heavy books (like slow JSON files) is too expensive.

**Apache Thrift** solves this by:
1.  **Providing a Contract**: A standard "order form" everyone understands.
2.  **Vacuum Packing**: Compressing the data into a tiny binary format that's super cheap and fast to ship.

## 1. What is Apache Thrift?

Thrift is a system used to define data types and services. 

It supports basic types like:
- `bool`: True/False
- `i32/i64`: Whole numbers (integers)
- `string`: Text
- `list`: A collection of items

We define these in a `.thrift` file, which acts as our **Contract**.

## 2. Step 1: The Contract (The Schema)

Before we ship anything, we need to agree on what a "Person" looks like. 

**The Analogy: The Menu Numbers**  
In a restaurant, instead of saying "I want a double-patty beef burger with extra cheese and no onions," you just say **"I'll take a Number 1."** It's much faster. 

In the code below, notice the numbers `1:`, `2:`, and `3:`. These are our **Field Tags** (Menu Numbers). Thrift uses these numbers to identify data instead of using long names, making it incredibly fast.

In [ ]:
%%writefile ../schema/person.thrift

# struct: a data model that defines the structure of a data object (like a class in Python)
struct Person {
  1: required string userName,
  2: optional i64 favoriteNumber,
  3: optional list<string> interests
  4: optional i32 grade                # new field added to the Person struct (optional exercise)
}

# service: a function (what the server can do) that defines the operations that can be performed on the data model (Person)
service School {
    Person teachCourse(1: required Person person, 2: required string course)
    Person assignGrade(1: required Person person, 2: required i32 grade)   # newly added to School service (optional exercise)
}


# What it does: It creates a .thrift file (person.thrift). This file is the blueprint or the rulebook.

# The Data (struct Person): It defines exactly what a "Person" looks like. 
# It says a person must have a userName (string), optional to add favoriteNumber (integer) and interests (list of strings).
# service School is the container. It is the contract that says, "I am a School service."
# Person teachCourse(...) is the function. It is the specific action you can ask this service to perform.
# Note: A service can have many functions. 
# For example, you could add Person graduateStudent(...) or string getSchoolName() inside the service School { ... } block

Overwriting ../schema/person.thrift


## 3. Step 2: The Kitchen (The Server)

Now we need someone to actually *do* something with the data. 

**The Analogy: The Chef**  
The Server is like a Chef in a kitchen. They sit there waiting for an order (an RPC call). When an order comes in, they follow the instructions and send the result back.

In our code, the `teachCourse` method is the Chef's skill: they take a person, add a new interest to their list, and return the updated person.

In [ ]:

%%writefile ../person_thrift_server.py  # Save the script to a file

import thriftpy2   # Load the engine

person_thrift = thriftpy2.load("./schema/person.thrift", module_name="person_thrift") # Load the blueprint (person.thrift) into Python so we can use it to create a server

from thriftpy2.rpc import make_server  # Load the engine that will create the server

# self: A standard Python reference to the class itself
# person: object contains userName, favoriteNumber, and interests listed in Thrift schema
# course: A string (like "Python 101") sent by the client.
# .append(course): toadd the new course to the end of that list (e.g., ['Math', 'History', 'Python 101']).

class School(object):
    def teachCourse(self, person, course):   
        # The 'Chef' adds a new skill to the person's interests
        person.interests.append(course)
        return person

# --- NEW METHOD ADDED HERE ---
    def assignGrade(self, person, grade):
        # The 'Chef' assigns the grade to the person
        person.grade = grade
        return person

server = make_server(person_thrift.School, School(), client_timeout=None)  # starting the Server
print("Chef is in the kitchen (Server started)...")
print("Press Ctrl+C to stop the server.")
server.serve()


# What it does: It creates the actual Python code (person_thrift_server.py) that does the work


Overwriting ../person_thrift_server.py


### How to run the Kitchen

1. Use the cell below to copy the command and paste on a new terminal.
2. Make sure your terminal are in the folder `5m-data-2.3-data-encoding-creation-flow`
3. Make sure you are in the right environment using `conda activate bde`.

Then, run `python person_thrift_server.py` in a new terminal. This will start the server. 

Once you see "Chef is in the kitchen," come back here to place your order!

## 4. Step 3: The Customer (The Client)

Now we act as the customer. We will create a person and send them to the kitchen to learn something new.

**The Analogy: Placing the Order**  
When you call `school.teachCourse(...)`, it feels like you're doing it locally on your computer. But behind the scenes, Thrift is **packing** your data into a box, **shipping** it to the server, and **bringing back** the result.

In [14]:
import thriftpy2
person_thrift = thriftpy2.load("../schema/person.thrift", module_name="person_thrift")

from thriftpy2.rpc import make_client

# Connect to the kitchen
school = make_client(person_thrift.School, timeout=None)

In [15]:
# Create a person named Martin
martin = person_thrift.Person(
    userName="Martin", favoriteNumber=1337, interests=["daydreaming", "hacking"]
)
print(f"Initial interests: {martin.interests}")

Initial interests: ['daydreaming', 'hacking']


In [16]:
# Send Martin to school to learn 'coding' remotely!
martin = school.teachCourse(martin, "coding")

In [17]:
# Let's see if he learned it
print(f"Updated interests: {martin.interests}")

Updated interests: ['daydreaming', 'hacking', 'coding']


## (Optional) 🎓 Exercises

Try these to see if you've mastered the "Courier Service":

1.  **Update the Contract**: Go back to the `.thrift` cell. Add `4: optional i32 grade` to the Person struct. 
2.  **Add a Skill**: Add a new method `assignGrade` to the School service. 
3.  **Update the Chef**: Update the Server code to handle the new `assignGrade` command.
4.  **Restart**: Remember, if you change the contract, the Chef and the Customer both need to re-read it! Stop your terminal and start the server again.

In [18]:
# Create a new Person named Sandy
sandy = person_thrift.Person(
    userName="Sandy", 
    favoriteNumber=7, 
    interests=["Math", "Science"]
)

print(f"Before: {sandy}")


Before: Person(userName='Sandy', favoriteNumber=7, interests=['Math', 'Science'], grade=None)


In [19]:
# Place the order! Ask the Chef to assign a grade to Sandy
# We are calling the NEW method you just added
import thriftpy2
from thriftpy2.rpc import make_client

# Reload the contract to ensure the new assignGrade method is recognized
person_thrift = thriftpy2.load("../schema/person.thrift", module_name="person_thrift")

# Re-establish the connection to the server
school_client = make_client(person_thrift.School, timeout=None)

updated_sandy = school_client.assignGrade(sandy, 95)

In [20]:
# Show the results
print("\n--- Order Complete ---")
print(f"Name: {updated_sandy.userName}")
print(f"Interests: {updated_sandy.interests}")
print(f"New Grade: {updated_sandy.grade}") # This is the new field!


--- Order Complete ---
Name: Sandy
Interests: ['Math', 'Science']
New Grade: 95


In [21]:
# 1. First, let's add a new course using the teachCourse method
sandy_with_course = school_client.teachCourse(sandy, "Art History")

# 2. Now, let's assign a grade using the assignGrade method
# We pass the updated Sandy object to the next call
final_sandy = school_client.assignGrade(sandy_with_course, 95)

print(final_sandy)

Person(userName='Sandy', favoriteNumber=7, interests=['Math', 'Science', 'Art History'], grade=95)


### 💡💡💡 A real-life data engineering scenario using Apache Thrift is a **cross-language data pipeline** where different systems need to communicate efficiently💡💡💡

Here is a concrete example, followed by how major tech companies implement it.

### 🏗️ The Scenario: A Cross-Language Data Pipeline

Imagine a company where data is generated by applications in different languages. You need a reliable way for them to talk to each other without building custom parsers for every pair.

*   **The Data Producer (Python)**: A Python service processes raw events and creates a `UserProfile` object containing a `userID` (integer), `userName` (string), and a list of `interests`.
*   **The Data Consumer (Java/Go)**: A Java service needs to receive this profile, validate it, and store it in a database. 

Without a shared framework, you'd face the "Language Barrier" from your image. **Apache Thrift solves this by acting as the universal "Contract"**:

1.  **Define the Contract**: You write a `.thrift` file defining the `UserProfile` structure. This is the single source of truth.
2.  **Generate Code**: Thrift's compiler automatically generates the necessary client and server code in both Python and Java from your `.thrift` file.
3.  **Communicate**: The Python producer serializes the `UserProfile` into Thrift's compact binary format and sends it. The Java consumer receives it, deserializes it back into a Java object, and processes it, all without the developers writing any serialization code.

### 🔧 Why Data Engineers Choose Thrift for This

Beyond just cross-language support, Thrift offers specific advantages for data infrastructure:

*   **High Throughput & Low Latency**: Thrift's binary protocol is highly efficient. Studies show it achieves excellent throughput and low latency, making it suitable for high-volume data pipelines. Meta notes its serialization is performant because the data model is strictly typed.
*   **Schema Evolution**: Thrift allows you to add new fields (like the `grade` field in your example) without breaking existing clients. The generated parsers are designed to handle missing, unknown, or reordered fields gracefully. This is critical for long-running data pipelines that must evolve over time.
*   **Compact Data Format**: Compared to verbose formats like JSON, Thrift's binary "vacuum packing" significantly reduces the size of data sent over the network or stored on disk, saving bandwidth and storage costs.

### 🌍 Real-World Implementations

Major tech companies have used Thrift at massive scale for years:

*   **Twitter's Data Pipelines**: Twitter's ad engagement analytics platform ingested data streamed from various sources into HDFS as **LZO-compressed Thrift files** within the Elephant Bird container format. These files were then processed by Scalding (a Scala-based data transformation library), demonstrating Thrift's use as a reliable, efficient on-disk format for batch processing.
*   **Meta (Facebook)**: Thrift is the "standard communication framework" across Meta's infrastructure, supporting hundreds of billions of queries per second (QPS). They rely on it for its **language interoperability** and **performance** across a wide range of services.
*   **Spark Thrift Server**: This is a widely used component in the modern data stack. It allows data engineers to submit SQL queries from tools like dbt or BI dashboards to a Spark cluster using the **Thrift protocol** as the connection method.
*   **Hive Metastore**: Many data platforms like DataHub use the **Thrift API** to connect to the Hive Metastore (HMS) to extract metadata about tables, schemas, and storage lineage.

In short, Apache Thrift is a foundational tool for building the reliable, high-performance, and language-agnostic data infrastructure that powers modern data platforms.